# Dyslexia Project — Full Retrain (Models 1, 2, 3)
## Fixed test set — all models evaluated on the same 36 pairs
### Run one MODEL BLOCK per Colab session to avoid disconnection
- **Session 1** → Run SETUP + MODEL 1 block (~30 min)
- **Session 2** → Run SETUP + MODEL 2 block (~50 min)
- **Session 3** → Run SETUP + MODEL 3 block (~55 min)
- **Session 4** → Run EVALUATION block for all 3 models (~45 min)

In [ ]:
# Cell 0 — Official Unsloth install (from docs)
!pip install unsloth
!pip install -q git+https://github.com/feralvam/easse.git
!pip install -q sacrebleu
print("Done — restarting now...")
import os
os.kill(os.getpid(), 9)

In [ ]:
# Cell 1 — Imports
import unsloth
from unsloth import FastLanguageModel
import torch, json, shutil
print("✅ Unsloth imported successfully")

In [ ]:
# Cell 2 — Mount Drive and copy all dataset files
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

# Copy all split files
files_to_copy = [
    'dataset_m1_train.jsonl',
    'dataset_m1_val.jsonl',
    'dataset_m2_train.jsonl',
    'dataset_m2_val.jsonl',
    'dataset_m3_train.jsonl',
    'dataset_m3_val.jsonl',
    'dataset_test_fixed.jsonl',
]
for f in files_to_copy:
    shutil.copy(DRIVE_PATH + f, '/content/')
    print(f'✅ Copied: {f}')

print('\nAll files ready.')

In [ ]:
# Cell 3 — Data loading function + prompt formatter
import json

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

def format_prompt(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    example['text'] = prompt
    return example

class SimpleDataset:
    def __init__(self, data):
        self.data = [{'text': d['text']} for d in data]
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]
    def map(self, func, **kwargs):
        self.data = [func(d) for d in self.data]
        return self
    @property
    def column_names(self):
        return list(self.data[0].keys()) if self.data else []

print('✅ Functions defined')

In [ ]:
# Cell 4 — Training function (reusable for all 3 models)
from trl import SFTTrainer, SFTConfig
from datasets import Dataset as HFDataset
from transformers import DataCollatorForLanguageModeling

def train_model(train_path, val_path, save_folder, model_name):
    print(f'\n{"="*60}')
    print(f'Training {model_name}')
    print(f'{"="*60}')

    # Load data
    train_data = [format_prompt(ex)['text'] for ex in load_jsonl(train_path)]
    val_data   = [format_prompt(ex)['text'] for ex in load_jsonl(val_path)]
    print(f'Train: {len(train_data)} pairs | Val: {len(val_data)} pairs')

    # Load fresh base model
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name='unsloth/mistral-7b-instruct-v0.2-bnb-4bit',
        max_seq_length=1024,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'right'

    # Configure LoRA
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable parameters: {trainable:,}')

    # Pre-tokenize — gives fix_untrained_tokens the input_ids it needs
    def tokenize(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            max_length=1024,
            padding=False,
        )

    train_hf = HFDataset.from_dict({'text': train_data})
    val_hf   = HFDataset.from_dict({'text': val_data})

    train_tok = train_hf.map(tokenize, batched=True, remove_columns=['text'])
    val_tok   = val_hf.map(tokenize,   batched=True, remove_columns=['text'])
    #train_tok.set_format('torch')
    #val_tok.set_format('torch')

    print(f'Tokenized. Train batches: {len(train_tok)}')

    # Training config — use TrainingArguments, not SFTConfig, to bypass Unsloth's SFT pipeline
    from transformers import TrainingArguments, Trainer

    training_args = TrainingArguments(
        output_dir=f'./{save_folder}_checkpoints',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=10,
        fp16=True,
        logging_steps=5,
        save_steps=30,
        eval_strategy='steps',
        eval_steps=30,
        load_best_model_at_end=True,
        report_to='none',
        optim='adamw_8bit',
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )

    # Train
    import time
    t0 = time.time()
    trainer.train()
    elapsed = (time.time() - t0) / 60
    print(f'\nTraining time: {elapsed:.1f} minutes')

    # Save to Drive
    save_path = DRIVE_PATH + save_folder + '/'
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'✅ {model_name} saved to Drive: {save_path}')

    return model, tokenizer

print('✅ train_model() function defined')

In [ ]:
# Cell 5 — Evaluation function (BLEU + SARI on fixed test set)
from sacrebleu.metrics import BLEU
from easse.sari import corpus_sari

INSTRUCTION = (
    'Tu es un expert en orthophonie et en éducation inclusive. '
    'Simplifie le texte suivant pour un enfant dyslexique francophone âgé de 6 à 8 ans. '
    'Applique ces règles obligatoires : '
    '(1) Phrases courtes de 8 à 10 mots maximum. '
    '(2) Structure Sujet + Verbe + Complément uniquement. '
    '(3) Vocabulaire simple — remplace les mots de plus de 3 syllabes. '
    '(4) Découpage syllabique avec tirets : ma-man, jar-din, é-lè-ve. '
    '(5) Conserve TOUS les détails narratifs. '
    '(6) Retour à la ligne à chaque phrase.'
)

def simplify_text(model, tokenizer, text, max_new_tokens=600):
    prompt = f'### Instruction:\n{INSTRUCTION}\n\n### Input:\n{text}\n\n### Response:\n'
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=1024  # ← increased
    ).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,  # ← increased to 600
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True)
    if '### ' in result:
        result = result[:result.index('### ')].strip()
    return result

def evaluate_on_fixed_test(model, tokenizer, model_name):
    print(f'\n{"="*60}')
    print(f'Evaluating {model_name} on fixed test set (36 pairs)')
    print(f'{"="*60}')

    # Switch to inference mode
    model = FastLanguageModel.for_inference(model)

    test_data  = load_jsonl('/content/dataset_test_fixed.jsonl')
    bleu_metric = BLEU(effective_order=True)

    orig_sents = []
    ref_sents  = []
    hyp_sents  = []
    references = []
    hypotheses = []

    for i, example in enumerate(test_data):
        orig = example['input']
        ref  = example['output']
        hyp  = simplify_text(model, tokenizer, orig)

        orig_sents.append(orig)
        ref_sents.append(ref)
        hyp_sents.append(hyp)
        references.append(ref)
        hypotheses.append(hyp)

        print(f'Pair {i+1:2d} | {example["id"]} | {example["theme"][:35]}')

    bleu_score = bleu_metric.corpus_score(hypotheses, [references])
    sari_score = corpus_sari(
        orig_sents=orig_sents,
        sys_sents=hyp_sents,
        refs_sents=[ref_sents],
    )

    print(f'\n{"="*50}')
    print(f'{model_name} — Fixed Test Set (36 pairs)')
    print(f'  BLEU : {bleu_score.score:.2f}')
    print(f'  SARI : {sari_score:.2f}')
    print(f'{"="*50}')

    return bleu_score.score, sari_score

print('✅ evaluate_on_fixed_test() function defined')

In [ ]:
test_data = load_jsonl(DRIVE_PATH + 'dataset_test_fixed.jsonl')
print(f'Test pairs: {len(test_data)}')

---
## 🔵 SESSION 1 — Train Model 1 (99 train + 10 val)
**Run Cell 0 → Cell 5 first, then this cell.**
Expected time: ~30 minutes total.

In [ ]:
# Cell 6 — Train Model 1 (99 pairs)
# ⚠️ Run this in a fresh session after Cell 0–5

model1, tokenizer1 = train_model(
    train_path='/content/dataset_m1_train.jsonl',
    val_path='/content/dataset_m1_val.jsonl',
    save_folder='model_v1_retrained',
    model_name='Model 1 (99 pairs)'
)
print('\n✅ Model 1 training complete and saved to Drive')

In [ ]:
# Cell 7 — Evaluate Model 1 on fixed test set
bleu1, sari1 = evaluate_on_fixed_test(model1, tokenizer1, 'Model 1 (99 pairs)')

# Save results
import json
results = {'model': 'Model 1', 'train_pairs': 99, 'bleu': bleu1, 'sari': sari1}
with open(DRIVE_PATH + 'results_model1.json', 'w') as f:
    json.dump(results, f)
print('✅ Results saved to Drive')

# Save Model 1 to Drive
#save_path = DRIVE_PATH + 'model_v1_retrained/'
#model1.save_pretrained(save_path)
#tokenizer1.save_pretrained(save_path)
#print(f'✅ Model 1 saved to {save_path}')

---
## 🟢 SESSION 2 — Train Model 2 (243 train + 27 val)
**Start a NEW Colab session. Run Cell 0 → Cell 5 first, then this cell.**
Expected time: ~50 minutes total.

In [ ]:
# Cell 8 — Train Model 2 (243 pairs)
# ⚠️ Run this in a fresh session after Cell 0–5

model2, tokenizer2 = train_model(
    train_path='/content/dataset_m2_train.jsonl',
    val_path='/content/dataset_m2_val.jsonl',
    save_folder='model_v2_retrained',
    model_name='Model 2 (243 pairs)'
)
print('\n✅ Model 2 training complete and saved to Drive')

In [ ]:
# Cell 9 — Evaluate Model 2 on fixed test set
bleu2, sari2 = evaluate_on_fixed_test(model2, tokenizer2, 'Model 2 (243 pairs)')

results = {'model': 'Model 2', 'train_pairs': 243, 'bleu': bleu2, 'sari': sari2}
with open(DRIVE_PATH + 'results_model2.json', 'w') as f:
    json.dump(results, f)
print('✅ Results saved to Drive')

---
## 🟠 SESSION 3 — Train Model 3 (294 train + 32 val)
**Start a NEW Colab session. Run Cell 0 → Cell 5 first, then this cell.**
Expected time: ~55 minutes total.

In [ ]:
# Cell 10 — Train Model 3 (294 pairs)
# ⚠️ Run this in a fresh session after Cell 0–5

model3, tokenizer3 = train_model(
    train_path='/content/dataset_m3_train.jsonl',
    val_path='/content/dataset_m3_val.jsonl',
    save_folder='model_v3_retrained',
    model_name='Model 3 (294 pairs)'
)
print('\n✅ Model 3 training complete and saved to Drive')

In [ ]:
# Cell 11 — Evaluate Model 3 on fixed test set
bleu3, sari3 = evaluate_on_fixed_test(model3, tokenizer3, 'Model 3 (294 pairs)')

results = {'model': 'Model 3', 'train_pairs': 294, 'bleu': bleu3, 'sari': sari3}
with open(DRIVE_PATH + 'results_model3.json', 'w') as f:
    json.dump(results, f)
print('✅ Results saved to Drive')

---
## 📊 SESSION 4 — Final Results Summary
**Run after all 3 models are trained and evaluated.**

In [ ]:
# Cell 12 — Load and display all results
import json
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

models = ['results_model1.json', 'results_model2.json', 'results_model3.json']
print(f'\n{"="*60}')
print(f'FINAL RESULTS — Fixed Test Set (36 pairs)')
print(f'{"="*60}')
print(f'{"Model":<25} {"Train Pairs":>12} {"BLEU":>8} {"SARI":>8}')
print('-' * 60)

for fname in models:
    try:
        with open(DRIVE_PATH + fname) as f:
            r = json.load(f)
        print(f'{r["model"]:<25} {r["train_pairs"]:>12} {r["bleu"]:>8.2f} {r["sari"]:>8.2f}')
    except:
        print(f'{fname} — not yet available')

print('-' * 60)
print('Zero-shot Mistral              —        26.85    53.01')
print('ETR-fr / Mistral+LoRA        523           —    42.27')
print('WiViCo / FLAN-T5          46,525      ~38-42       —')

In [ ]:
# Cell 13 — Inference and Readability Analysis of Model Outputs
import json, re, gc, torch
from unsloth import FastLanguageModel

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

# Load test set
test_data = load_jsonl(DRIVE_PATH + 'dataset_test_fixed.jsonl')
print(f'Test pairs: {len(test_data)}')

# ── Inference ──────────────────────────────────────────────────────
def generate_outputs(model_path, model_label, test_data):
    print(f'\nLoading {model_label}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=1024, dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.for_inference(model)
    tokenizer.pad_token = tokenizer.eos_token

    results = []
    for i, ex in enumerate(test_data):
        # ✅ uses simplify_text — correct prompt, stops at ### Response:
        text = simplify_text(model, tokenizer, ex['input'])
        results.append({
            'id':               ex['id'],
            'niveau':           ex['niveau'],
            'original':         ex['input'],
            'human_simplified': ex['output'],
            'model_output':     text,
        })
        print(f'  {i+1}/36 | {ex["id"]} | empty={not text.strip()}')

    empty = sum(1 for r in results if not r['model_output'].strip())
    print(f'Empty outputs: {empty}/36')

    out_path = DRIVE_PATH + f'outputs_{model_label}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'✅ {model_label} saved to {out_path}')

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return results

# ── Run all 3 models ───────────────────────────────────────────────
outputs1 = generate_outputs(DRIVE_PATH + 'model_v1_retrained/', 'model1', test_data)
outputs2 = generate_outputs(DRIVE_PATH + 'model_v2_retrained/', 'model2', test_data)
outputs3 = generate_outputs(DRIVE_PATH + 'model_v3_retrained/', 'model3', test_data)

# ── Readability metrics ────────────────────────────────────────────
def split_sentences(text):
    return [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

def count_syllables_fr(word):
    word = word.lower().strip(".,!?;:\"'«»()-")
    if not word: return 0
    if len(word) > 2 and word.endswith('e') and word[-2] not in 'aeiouéèêëàâùûîïôœ':
        word = word[:-1]
    vowels = 'aeiouyéèêëàâùûîïôœæ'
    count, prev_vowel = 0, False
    for ch in word:
        is_vowel = ch in vowels
        if is_vowel and not prev_vowel: count += 1
        prev_vowel = is_vowel
    return max(1, count)

def tokenize_words(text):
    text = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)
    return re.findall(r"[a-zA-ZÀ-ÿ']+", text.lower())

def compute_metrics(text):
    text_clean = re.sub(r'(?<=[a-zA-ZÀ-ÿ])-(?=[a-zA-ZÀ-ÿ])', '', text)
    sents = split_sentences(text)
    words = tokenize_words(text)
    chars = len(re.sub(r'\s', '', text_clean))
    if not sents or not words:
        return {}
    asl = len(words) / len(sents)
    asw = sum(count_syllables_fr(w) for w in words) / len(words)
    return {
        'flesch':       round(207 - 1.015*asl - 73.6*asw, 2),
        'gunning_fog':  round(0.4*(asl + len([w for w in words if count_syllables_fr(w)>=3])/len(words)*100), 2),
        'ari':          round(4.71*(chars/len(words)) + 0.5*(len(words)/len(sents)) - 21.43, 2),
        'avg_sent_len': round(asl, 2),
        'simple_ratio': round(len([w for w in words if count_syllables_fr(w)<=2])/len(words), 4),
    }

def avg_metrics(outputs, field):
    all_m = [compute_metrics(ex[field]) for ex in outputs]
    all_m = [m for m in all_m if m]  # filter empty
    if not all_m: return {}
    keys = all_m[0].keys()
    return {k: round(sum(m[k] for m in all_m)/len(all_m), 2) for k in keys}

# ── Build comparison table ─────────────────────────────────────────
comparison = {
    'original':         avg_metrics(outputs3, 'original'),
    'human_simplified': avg_metrics(outputs3, 'human_simplified'),
    'model1':           avg_metrics(outputs1, 'model_output'),
    'model2':           avg_metrics(outputs2, 'model_output'),
    'model3':           avg_metrics(outputs3, 'model_output'),
}

with open(DRIVE_PATH + 'readability_comparison.json', 'w', encoding='utf-8') as f:
    json.dump(comparison, f, indent=2)

print('\n' + '='*70)
print(f"{'Metric':<20} {'Original':>9} {'Human':>9} {'M1':>9} {'M2':>9} {'M3':>9}")
print('-'*70)
for m in ['flesch','gunning_fog','ari','avg_sent_len','simple_ratio']:
    vals = [comparison[k][m] for k in ['original','human_simplified','model1','model2','model3']]
    print(f"{m:<20} " + " ".join(f"{v:>9}" for v in vals))

print('\n✅ Saved to readability_comparison.json')

In [ ]:
import json

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

for label in ['model1', 'model2', 'model3']:
    data = json.load(open(DRIVE_PATH + f'outputs_{label}.json', encoding='utf-8'))
    empty = sum(1 for ex in data if not ex['model_output'].strip())
    print(f'{label}: {len(data)} total, {empty} empty, {len(data)-empty} valid')
    # Show first valid output
    for ex in data:
        if ex['model_output'].strip():
            print(f'  Sample: {ex["model_output"][:100]}')
            break

---
## ## 🟢 SESSION 5 — Train 3 other models to get the CLEU learning curve
**Start a NEW Colab session. Run Cell 0 → Cell 5 first, then this cell.**
Expected time: ~50 minutes total.

Model 50 (50 train + 5 val)
Model 100 (100 train + 10 val)
Model 150 (150 train + 15 val)

In [ ]:
## TRAIN the 3 other models
# Cell 14a — Train LC50
model_lc50, tokenizer_lc50 = train_model(
    train_path=DRIVE_PATH + 'dataset_lc50_train.jsonl',
    val_path=DRIVE_PATH + 'dataset_lc50_val.jsonl',
    save_folder='model_lc50',
    model_name='LC50 (50 pairs)'
)
model_lc50.save_pretrained(DRIVE_PATH + 'model_lc50/')
tokenizer_lc50.save_pretrained(DRIVE_PATH + 'model_lc50/')
print('✅ LC50 saved')
del model_lc50, tokenizer_lc50
import gc, torch; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 14b — Train LC150
model_lc150, tokenizer_lc150 = train_model(
    train_path=DRIVE_PATH + 'dataset_lc150_train.jsonl',
    val_path=DRIVE_PATH + 'dataset_lc150_val.jsonl',
    save_folder='model_lc150',
    model_name='LC150 (150 pairs)'
)
model_lc150.save_pretrained(DRIVE_PATH + 'model_lc150/')
tokenizer_lc150.save_pretrained(DRIVE_PATH + 'model_lc150/')
print('✅ LC150 saved')
del model_lc150, tokenizer_lc150
import gc, torch; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 14c — Train LC200
model_lc200, tokenizer_lc200 = train_model(
    train_path=DRIVE_PATH + 'dataset_lc200_train.jsonl',
    val_path=DRIVE_PATH + 'dataset_lc200_val.jsonl',
    save_folder='model_lc200',
    model_name='LC200 (200 pairs)'
)
model_lc200.save_pretrained(DRIVE_PATH + 'model_lc200/')
tokenizer_lc200.save_pretrained(DRIVE_PATH + 'model_lc200/')
print('✅ LC200 saved')
del model_lc200, tokenizer_lc200
import gc, torch; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 15 — Evaluate LC models (BLEU + SARI only)
import json, torch
from unsloth import FastLanguageModel
from sacrebleu.metrics import BLEU
from easse.sari import corpus_sari

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

def evaluate_lc_model(model_path, label):
    print(f'\nLoading {label}...')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=1024, dtype=None, load_in_4bit=True,
    )
    model = FastLanguageModel.for_inference(model)
    tokenizer.pad_token = tokenizer.eos_token

    test_data = load_jsonl(DRIVE_PATH + 'dataset_test_fixed.jsonl')
    bleu_metric = BLEU(effective_order=True)

    orig_sents, ref_sents, hyp_sents = [], [], []

    for i, ex in enumerate(test_data):
        orig = ex['input']
        ref  = ex['output']
        hyp  = simplify_text(model, tokenizer, orig)
        orig_sents.append(orig)
        ref_sents.append(ref)
        hyp_sents.append(hyp)
        if (i+1) % 10 == 0:
            print(f'  {i+1}/36 done')

    bleu = bleu_metric.corpus_score(hyp_sents, [ref_sents]).score
    sari = corpus_sari(
        orig_sents=orig_sents,
        sys_sents=hyp_sents,
        refs_sents=[ref_sents],
    )
    print(f'✅ {label} → BLEU: {bleu:.2f}  SARI: {sari:.2f}')

    import gc, torch
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    return {'n_pairs': int(label.replace('LC','')), 'bleu': round(bleu,2), 'sari': round(sari,2)}

# Run all 3
results_lc50  = evaluate_lc_model(DRIVE_PATH + 'model_lc50/',  'LC50')
results_lc150 = evaluate_lc_model(DRIVE_PATH + 'model_lc150/', 'LC150')
results_lc200 = evaluate_lc_model(DRIVE_PATH + 'model_lc200/', 'LC200')

# Combine with known results
known = [
    {'n_pairs': 99,  'bleu': 61.53, 'sari': 79.50},
    {'n_pairs': 243, 'bleu': 68.29, 'sari': 83.09},
    {'n_pairs': 294, 'bleu': 70.02, 'sari': 84.13},
]
all_results = sorted(known + [results_lc50, results_lc150, results_lc200],
                     key=lambda x: x['n_pairs'])

# Save
with open(DRIVE_PATH + 'learning_curve.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Print
print('\n' + '='*40)
print(f"{'Pairs':>8} {'BLEU':>8} {'SARI':>8}")
print('-'*40)
for r in all_results:
    print(f"{r['n_pairs']:>8} {r['bleu']:>8.2f} {r['sari']:>8.2f}")
print('\n✅ Saved to learning_curve.json')

In [ ]:
import json

DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks/dyslexie_projet/'

for fname in ['outputs_model3.json', 'outputs_model3_fixed.json']:
    data = json.load(open(DRIVE_PATH + fname, encoding='utf-8'))
    empty = sum(1 for ex in data if not ex['model_output'].strip())
    sample = next((ex['model_output'][:80] for ex in data if ex['model_output'].strip()), 'ALL EMPTY')
    print(f'{fname}: {len(data)} total, {empty} empty')
    print(f'  Sample: {sample}')
    print()